# Vytvoření souhrnné kostky výnosového auditu telekomunikačního operátora pomocí PROC SUMMARY

## Shrnutí

Tým výnosového auditu (revenue assurance) bezdrátového operátora předagreguje měsíc fakturačních záznamů za účastníka a den do kompaktní souhrnné kostky, aby analytici mohli prokliknout vyúčtovanou tržbu podle regionu, tarifu a typu volání bez opětovného procházení surové faktové tabulky. `PROC SUMMARY` sroluje 100 záznamů za účastníka a den do vícerozměrné sady buněk: nejjemnější zrno (region x tarif x typ volání) zaplní 25 z 27 možných buněk, zatímco pojmenované podkostky poskytují okrajové součty, které analytici dotazují nejčastěji. V tomto ukázkovém měsíci operátor vyúčtoval \$222,78 napříč třemi aktivními regiony, přičemž Jih (\$97,44) a Východ (\$86,94) společně tvoří 83 % tržby a Sever zaostává s \$38,40. Nejbohatší jednotlivou podkostkou je hlasová služba tarifu Plus (\$59,06 přes 18 záznamů) a seřazení buněk podle výnosu na záznam ukazuje buňky datového provozu jako nejhodnotnější cíle pro audit úniků (nejvyšší výnos \$7,87/záznam). Každé číslo níže je čteno přímo z provedeného výstupu.

## Zdroje dat

| Datová sada | Řádky | Zrno | Klíčové proměnné |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | Jeden řádek na souhrn využití účastníka za den | `region` (Východ/Jih/Sever), `plan_tier` (Předplacená/Základní/Plus), `call_type` (Hlas/SMS/Data), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | Jeden řádek na neprázdnou buňku (region x plan_tier x call_type) | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | Jeden řádek na buňku pojmenovaných podkostek pro prokliky | `_TYPE_`, `_FREQ_`, `rev_total` |

Všechna data jsou generována přímo v sešitu pomocí `call streaminit()` / `rand()`; nejsou vyžadovány žádné externí soubory ani přístup k síti. Toto prostředí běží bez licence, takže zapsané datové sady jsou omezeny na 100 pozorování - scénář je navržen tak, aby byla kostka v rámci tohoto limitu plně zaplněna.

## Od surových záznamů o hovorech k prokliknutelné kostce

Bezdrátoví operátoři vyúčtovávají tržby napříč miliony denních událostí využití. Aby analytici výnosového auditu mohli odpovědět na otázky jako *"Jaká byla tržba z hlasových služeb tarifu Plus v regionu Jih minulý měsíc?"* bez opětovného procházení surové faktové tabulky pokaždé znovu, **předagregujeme** data do kompaktní souhrnné kostky.

`PROC SUMMARY` je nástroj Base SAS přesně pro tento účel: seskupí plochou faktovou tabulku podle jedné nebo více dimenzí `CLASS` a zapíše požadované statistiky do výstupní datové sady, přičemž každý řádek označí `_TYPE_` (které dimenze jsou aktivní) a `_FREQ_` (počet záznamů za buňkou). Tato výstupní datová sada *je* vícerozměrná kostka - stejná struktura souhrnů, jakou by vystavil nástroj OLAP, materializovaná jako obyčejná datová sada SAS, kterou lze tisknout, spojovat a prokrajovat.

Tento sešit:

1. Generuje realistický měsíc fakturačních záznamů za účastníka a den.
2. Sestaví nejjemnější kostku (region x tarif x typ volání) pomocí `PROC SUMMARY NWAY`.
3. Materializuje pojmenované **podkostky pro prokliky** pomocí příkazu `TYPES`.
4. Promítne tržbu na základnu účastníků pomocí `WEIGHT`, prokliká do jednoho regionu a seřadí buňky podle výnosu na záznam pro triáž úniků.

## Krok 1 - Generování syntetických dat o hovorech / fakturaci

Každý řádek shrnuje využití jedné služby jedním účastníkem za jeden den. Používáme `call streaminit` pro reprodukovatelnost a `rand()` k vygenerování věrohodných rozdělení: tržba a využití škálují podle tarifu, hlasová tržba sleduje fakturovatelné minuty a datová tržba sleduje megabajty. Každé `RAND('table', ...)` má jednu pravděpodobnost na kategorii, takže se ve vzorku 100 záznamů objeví každý region, tarif i typ volání. Je připojena malá průzkumná váha `subscriber_wt`, abychom později mohli ukázat váženou míru.

In [1]:
data work.cdr_billing;
    CALL streaminit(20260131);
    DÉLKA region $8 plan_tier $15 call_type $7 device_class $10 bill_month $7;
    ŠTÍTEK revenue       = "Vyúčtovaná tržba (USD)"
          call_minutes  = "Fakturovatelné hlasové minuty"
          data_mb       = "Objem dat (MB)"
          subscriber_wt = "Váha průzkumu účastníka";

    OPAKUJ i = 1 TO 100;
        /* --- Dimenze: jedna pravděpodobnost na úroveň, součet 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        VYBRAT (r);
            KDYŽ_V (1) region = "Východ";
            KDYŽ_V (2) region = "Jih";
            JINAK_V region = "Sever";
        KONEC;

        p = rand("table", 0.30, 0.40, 0.30);
        VYBRAT (p);
            KDYŽ_V (1) plan_tier = "Předplacená";
            KDYŽ_V (2) plan_tier = "Základní";
            JINAK_V plan_tier = "Plus";
        KONEC;

        c = rand("table", 0.50, 0.30, 0.20);
        VYBRAT (c);
            KDYŽ_V (1) call_type = "Hlas";
            KDYŽ_V (2) call_type = "SMS";
            JINAK_V call_type = "Data";
        KONEC;

        KDYŽ rand("uniform") < 0.55 PAK device_class = "Chytrý";
        JINAK device_class = "Klasický";

        bill_month = "2026-01";

        /* --- Míry, škálované podle tarifu a služby --- */
        tier_mult = (plan_tier = "Předplacená")*0.7
                  + (plan_tier = "Základní")*1.0
                  + (plan_tier = "Plus")*1.7;

        call_minutes = round((call_type = "Hlas")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Data")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        VÝSTUP;
    KONEC;
    ODSTRANIT i r p c tier_mult base_rev;
SPUSTIT;

PROCEDURA TISK data=work.cdr_billing(obs=8) ŠTÍTEK noobs;
    ŠTÍTEK region="Region" plan_tier="Tarif" call_type="Typ volání"
          device_class="Typ zařízení" bill_month="Fakturační měsíc";
    NÁZEV "Ukázkové záznamy vyúčtování účastníka za den";
SPUSTIT;

                                      Ukázkové záznamy vyúčtování účastníka za den                                      

 Region          Tarif    Typ volání     Typ zařízení      Fakturační měsíc    Fakturovatelné hlasové minuty  Objem dat (MB)      Vyúčtovaná tržba (USD)       Váha průzkumu účastníka
Sever    Plus           SMS           Chytrý           2026-01                                             0               0                        0.67                          1.13
Jih      Předplacená    SMS           Klasický         2026-01                                             0               0                        0.94                          1.42
Jih      Plus           SMS           Chytrý           2026-01                                             0               0                        0.99                          0.86
Jih      Plus           SMS           Chytrý           2026-01                                             0               0                      


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Krok 2 - Sestavení nejjemnější kostky pomocí PROC SUMMARY NWAY

`NWAY` ponechá pouze jedinou nejpodrobnější kombinaci všech dimenzí `CLASS` - zde každou zaplněnou buňku (region x plan_tier x call_type). Pro každou buňku ukládáme `SUM`, `MEAN` a `MAX` tržby plus celkové minuty a megabajty, takže analytik může přečíst celkovou tržbu na buňku, odvodit průměr na záznam a najít největší jednotlivý poplatek. `_FREQ_` zaznamenává, kolik záznamů za účastníka a den stojí za každou buňkou. `_TYPE_` zde vypouštíme, protože na úrovni zrna NWAY má každý řádek stejný typ.

In [2]:
PROCEDURA summary data=work.cdr_billing NWAY;
    TŘÍDA region plan_tier call_type;
    PROMĚNNÁ revenue call_minutes data_mb;
    VÝSTUP out=work.cube_nway(ODSTRANIT=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
SPUSTIT;

PROCEDURA TISK data=work.cube_nway(obs=12) noobs ŠTÍTEK;
    NÁZEV "Buňky kostky NWAY (region * tarif * typ volání)";
    ŠTÍTEK region="Region" plan_tier="Tarif" call_type="Typ volání"
          _freq_="Počet záznamů" rev_total="Tržba celkem" rev_mean="Průměrná tržba"
          rev_max="Max. tržba" min_total="Minuty celkem" data_total="Data celkem (MB)";
    FORMÁT rev_total rev_mean rev_max min_total data_total comma10.2;
SPUSTIT;

                                    Buňky kostky NWAY (region * tarif * typ volání)                                     

Region          Tarif    Typ volání     Počet záznamů   Tržba celkem      Průměrná tržba   Max. tržba  Minuty celkem  Data celkem (MB)
Jih     Plus           Data                         2          11.92                5.96        10.16           0.00          1,122.90
Jih     Plus           Hlas                         8          27.07                3.38         5.99         521.90              0.00
Jih     Plus           SMS                          5           5.16                1.03         1.35           0.00              0.00
Jih     Předplacená    Data                         5          12.34                2.47         4.79           0.00          1,170.40
Jih     Předplacená    Hlas                         3           2.59                0.86         1.89          49.90              0.00
Jih     Předplacená    SMS                          2           1.82


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Krok 3 - Materializace pojmenovaných podkostek pro prokliky pomocí TYPES

OLAP kostka předem ukládá souhrny, ke kterým analytici nejčastěji navigují. Příkaz `TYPES` dělá přesně to: každý termín požádá `PROC SUMMARY` o vydání jedné podkostky. Požadujeme celkový součet `()`, okrajový součet přes `region` a dvourozměrné podkostky `region*plan_tier` a `call_type*plan_tier` - proklikové cesty, které by vystavil dashboard tržeb. SAS označí každou podkostku kódem `_TYPE_` (bitová maska přes seznam `CLASS`), takže jedna výstupní datová sada nese každou úroveň hierarchie.

In [3]:
PROCEDURA summary data=work.cdr_billing;
    TŘÍDA region plan_tier call_type;
    PROMĚNNÁ revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    VÝSTUP out=work.cube_hier
           sum(revenue)=rev_total;
SPUSTIT;

PROCEDURA TISK data=work.cube_hier noobs ŠTÍTEK;
    NÁZEV "Souhrny podkostek: celkový součet, region, region*tarif, typ volání*tarif";
    PROMĚNNÁ _type_ region plan_tier call_type _freq_ rev_total;
    ŠTÍTEK _type_="Typ podkostky" region="Region" plan_tier="Tarif" call_type="Typ volání"
          _freq_="Počet záznamů" rev_total="Tržba celkem";
    FORMÁT rev_total comma10.2;
SPUSTIT;

                       Souhrny podkostek: celkový součet, region, region*tarif, typ volání*tarif                        

Typ podkostky   Region          Tarif    Typ volání     Počet záznamů   Tržba celkem
            0                                                     100         222.78
            3           Plus           Data                         3          17.46
            3           Plus           Hlas                        18          59.06
            3           Plus           SMS                         13          11.75
            3           Předplacená    Data                         7          14.50
            3           Předplacená    Hlas                        16          24.77
            3           Předplacená    SMS                          7           6.82
            3           Základní       Data                         8          38.06
            3           Základní       Hlas                        20          42.33
            3           Zákl


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Krok 4 - Vážená projekce, regionální proklik a triáž úniků

Tři čtení, která tým výnosového auditu skutečně provádí nad kostkou:

- **Vážená projekce.** Připojení `WEIGHT=subscriber_wt` k souhrnu `region*plan_tier` vykáže tržbu promítnutou na celou základnu účastníků, kterou vzorek reprezentuje, namísto surových vzorkovaných řádků.
- **Proklik.** Filtrování kostky NWAY na jeden region rozvine jednu větev hierarchie do detailu - zde region Jih - podle tarifu a služby.
- **Triáž úniků.** Seřazení buněk podle průměrné tržby na záznam ukáže buňky s nejvyšším výnosem; buňky s nízkou četností a vysokým výnosem jsou přesně to, co audit prověřuje kvůli chybně naceněné nebo unikající tržbě.

In [4]:
/* Vážená tržba promítnutá na základnu účastníků */
PROCEDURA summary data=work.cdr_billing NWAY;
    TŘÍDA region plan_tier;
    PROMĚNNÁ revenue;
    VÁHA subscriber_wt;
    VÝSTUP out=work.cube_wt(ODSTRANIT=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
SPUSTIT;

PROCEDURA TISK data=work.cube_wt noobs ŠTÍTEK;
    NÁZEV "Vážená tržba podle regionu * tarifu (promítnuto na základnu účastníků)";
    ŠTÍTEK region="Region" plan_tier="Tarif" rev_weighted="Vážená tržba" records="Počet záznamů";
    FORMÁT rev_weighted comma10.2;
SPUSTIT;

/* Proklik do větve regionu Jih v kostce */
PROCEDURA TISK data=work.cube_nway noobs ŠTÍTEK;
    KDE region = "Jih";
    NÁZEV "Detail regionu Jih: buňky tržeb podle tarifu a typu volání";
    PROMĚNNÁ plan_tier call_type _freq_ rev_total rev_mean;
    ŠTÍTEK plan_tier="Tarif" call_type="Typ volání" _freq_="Počet záznamů"
          rev_total="Tržba celkem" rev_mean="Průměrná tržba";
    FORMÁT rev_total rev_mean comma10.2;
SPUSTIT;

/* Seřazení buněk podle výnosu na záznam pro triáž úniků */
PROCEDURA ŘADIT data=work.cube_nway out=work.cube_ranked;
    PODLE SESTUPNĚ rev_mean;
SPUSTIT;

PROCEDURA TISK data=work.cube_ranked(obs=6) noobs ŠTÍTEK;
    NÁZEV "Buňky s nejvyšším průměrným výnosem (výnos na záznam)";
    PROMĚNNÁ region plan_tier call_type _freq_ rev_mean rev_max;
    ŠTÍTEK region="Region" plan_tier="Tarif" call_type="Typ volání" _freq_="Počet záznamů"
          rev_mean="Průměrná tržba" rev_max="Max. tržba";
    FORMÁT rev_mean rev_max comma10.2;
SPUSTIT;

                         Vážená tržba podle regionu * tarifu (promítnuto na základnu účastníků)                         

 Region          Tarif      Vážená tržba     Počet záznamů
Jih      Plus                      56.29                15
Jih      Předplacená               27.77                10
Jih      Základní                  58.63                14
Sever    Plus                      22.42                 7
Sever    Předplacená               20.56                 9
Sever    Základní                  18.30                 7
Východ   Plus                      59.59                12
Východ   Předplacená               29.77                11
Východ   Základní                  50.85                15

                               Detail regionu Jih: buňky tržeb podle tarifu a typu volání                               

        Tarif    Typ volání     Počet záznamů   Tržba celkem      Průměrná tržba
Plus           Data                         2          11.92                5.96
Plu


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Interpretace výsledků

Souhrnná kostka promění 100 surových řádků za účastníka a den v kompaktní sadu předagregovaných buněk, které okamžitě zodpoví proklikové otázky, bez opětovného procházení faktové tabulky:

- **Kde je tržba.** Okrajový součet `region` (`_TYPE_=4`) ukazuje, že Jih vyúčtoval \$97,44 a Východ \$86,94 - společně 83 % z \$222,78 za měsíc - zatímco Sever přispěl \$38,40. V rámci podkostky `call_type*plan_tier` (`_TYPE_=3`) je hlasová služba tarifu Plus jedinou nejbohatší buňkou s \$59,06 přes 18 záznamů, hned za ní následuje hlasová služba tarifu Základní s \$42,33.
- **Vážená projekce.** Použití průzkumové váhy přeskupí pořadí směrem k tarifům, jejichž účastníci nesou větší váhu: Východ-Plus (\$59,59) a Jih-Základní (\$58,63) vedou v promítnuté tržbě `region*plan_tier` - jiný obrázek než nevážené regionální součty a připomínka, že při odhadu expozice je třeba vykazovat promítnutou, nikoli vzorkovanou tržbu.
- **Výnos na záznam a triáž úniků.** Seřazení buněk NWAY podle `rev_mean` staví na první místa buňky datového provozu - Sever-Základní-Data (\$7,87/záznam) a Jih-Plus-Data (\$5,96/záznam) - což potvrzuje, že intenzivní využívání dat pohání nejvyšší tržbu na záznam. Vedoucí buňky s nízkou četností (jeden nebo dva záznamy) jsou přesně ty, pro které by analytik výnosového auditu vytáhl podkladové záznamy o hovorech, aby ověřil, že vysoký poplatek je správně naceněn a nejde o chybu.

Pro tým výnosového auditu je tato kostka základem pro dashboardy odchylek: porovnat vyúčtovanou tržbu s očekávanou tržbou podle ceníku pro každou buňku (region, tarif, typ volání), přičemž buňky s největší odchylkou průměru nebo váženého součtu se stávají případy hodnými auditu. Protože celá struktura je obyčejná datová sada SAS, lze kostku dalšího měsíce sjednotit, porovnat rozdílem nebo spojit s ceníkem stejnými nástroji Base SAS.